In [1]:
%uv pip install --upgrade uv

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 202ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
Prepared 1 package in 373ms
Uninstalled 1 package in 3ms
Installed 1 package in 69ms
 - uv==0.5.14
 + uv==0.11.19
Note: you may need to restart the kernel to use updated packages.


In [2]:
%uv pip -q install openai opencv-python datasets

Note: you may need to restart the kernel to use updated packages.


# Import thư viện

In [1]:
import os
import io
import base64
import json
import numpy as np
import cv2
from PIL import Image
from datasets import load_dataset
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# Cấu hình tham số

In [2]:
# --- CẤU HÌNH HỆ THỐNG ---
HF_REPO_ID = "UngLong/radiology-test-v2"
VOLUME_IMAGE_NAME = "/mnt/ct" 
TARGET_SIZE = 896
MAX_SLICES = 85
MAX_CONCURRENT_REQUESTS = 2
OUTPUT_DIR = "result"

# Khởi tạo OPENAI

In [3]:
# Khởi tạo OpenAI Client kết nối tới Modal vLLM
client = OpenAI(
    base_url="https://minh0985362932--example-vllm-inference-serve.modal.run/v1",
    api_key="not-needed" 
)

# Tạo thư mục lưu kết quả nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hàm tiền xử lý ảnh

In [4]:
# --- CÁC HÀM TIỀN XỬ LÝ ẢNH (ON-THE-FLY) ---
def _hu_window(s: np.ndarray, wmin: float, wmax: float) -> np.ndarray:
    return ((np.clip(s, wmin, wmax) - wmin) / (wmax - wmin) * 255).astype(np.uint8)


def load_and_preprocess_slices(npy_path: str, max_slices: int) -> list[Image.Image]:
    """Load và tiền xử lý ảnh CT chuẩn 896x896 RGB giống lúc Finetune"""
    if not os.path.exists(npy_path):
        raise FileNotFoundError(f"Không tìm thấy file npy tại: {npy_path}")
        
    volume = np.load(npy_path).astype(np.float32)  # [Z, H, W]
    Z = volume.shape[0]
    indices = np.linspace(0, Z - 1, min(Z, max_slices), dtype=int)

    pil_slices = []
    for idx in indices:
        s = volume[idx]
        r = _hu_window(s, -1024.0, 1024.0)
        g = _hu_window(s, -135.0, 215.0)
        b = _hu_window(s, 0.0, 80.0)

        # Ghép 3 cửa sổ thành ma trận RGB
        rgb = np.stack([r, g, b], axis=-1)

        # Resize về hình vuông 896 bằng BILINEAR
        if rgb.shape[:2] != (TARGET_SIZE, TARGET_SIZE):
            rgb = cv2.resize(
                rgb, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LINEAR
            )

        # Chuyển thành ảnh PIL
        pil_slices.append(Image.fromarray(rgb, mode="RGB"))

    return pil_slices


def _encode_image_to_base64(pil_img: Image.Image) -> str:
    """Encode ảnh PIL sang chuỗi base64 định dạng JPEG"""
    format_str = "jpeg"
    with io.BytesIO() as img_bytes:
        pil_img.save(img_bytes, format=format_str)
        img_bytes.seek(0)
        encoded_string = base64.b64encode(img_bytes.getbuffer()).decode("utf-8")
    return f"data:image/{format_str};base64,{encoded_string}"


# --- HÀM XỬ LÝ CHÍNH CHO TỪNG CA BỆNH ---
# --- HÀM XỬ LÝ CHÍNH CHO TỪNG CA BỆNH ---
def process_single_case(example):
    """Xử lý on-the-fly cho một sample: Load -> Preprocess -> API -> Save"""
    pid = example['pids']
    key = example['keys']
    
    # Định nghĩa npy_rel_path là đường dẫn tương đối (ví dụ: "pid_01/key_01.npy")
    npy_rel_path = f"{pid}/{key}.npy"
    
    # 1. Định nghĩa phần OUTPUT FORMAT mới có chứa cấu trúc giải thích (explanation)
    new_output_format = """[OUTPUT FORMAT]
Respond ONLY with a single JSON object (no extra text outside the JSON). Each key must contain an object with two fields: "value" (either "No" or "Yes") and "explanation" (a string detailing the radiological findings that support the choice).
    
Follow this exact structure:
{
  "chest_abn_54": { "value": "...", "explanation": "..." },
  "chest_abn_55": { "value": "...", "explanation": "..." },
  "chest_abn_56": { "value": "...", "explanation": "..." },
  "chest_abn_57": { "value": "...", "explanation": "..." },
  "chest_abn_58": { "value": "...", "explanation": "..." },
  "chest_abn_59": { "value": "...", "explanation": "..." },
  "chest_abn_61": { "value": "...", "explanation": "..." },
  "nodule_presence": { "value": "...", "explanation": "..." }
}"""
    
    # 2. Lấy prompt gốc từ dữ liệu
    prompt_text = example["prompt"]
    
    # 3. Thay thế câu lệnh chung ở phần [QUESTIONS] để yêu cầu thêm giải thích
    prompt_text = prompt_text.replace(
        "For each item below, choose exactly one of its allowed values:",
        "For each item below, choose exactly one of its allowed values and provide a brief clinical explanation based on the CT scan findings:"
    )
    
    # 4. Tìm vị trí của [OUTPUT FORMAT] cũ và cắt bỏ nó, sau đó nối phần format mới vào
    if "[OUTPUT FORMAT]" in prompt_text:
        prompt_text = prompt_text.split("[OUTPUT FORMAT]")[0]
        prompt_text = prompt_text + new_output_format

    # 5. Tạo đường dẫn file đầu ra giữ nguyên cấu trúc thư mục
    txt_rel_path = npy_rel_path.replace(".npy", ".txt")
    output_file_path = os.path.join("result", txt_rel_path)
    
    # Lấy thư mục cha của file đầu ra và tự động tạo nếu chưa có
    output_dir = os.path.dirname(output_file_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # Bỏ qua nếu file đã được xử lý từ trước (Tránh chạy lại khi lỗi)
    if os.path.exists(output_file_path):
        return f"[BỎ QUA] {txt_rel_path} đã tồn tại."

    try:
        # Đường dẫn tuyệt đối tới file npy nguồn
        full_npy_path = os.path.join(VOLUME_IMAGE_NAME, npy_rel_path)
        
        # Xử lý ảnh on-the-fly
        pil_slices = load_and_preprocess_slices(full_npy_path, MAX_SLICES)
        
        # Tạo cấu trúc messages chuẩn OpenAI Vision API
        user_content = []
        for slice_num, slice_img in enumerate(pil_slices, 1):
            base64_url = _encode_image_to_base64(slice_img)
            user_content.append({
                "type": "image_url",
                "image_url": {
                    "url": base64_url,
                    "detail": "high"
                }
            })
            user_content.append({
                "type": "text",
                "text": f"SLICE {slice_num}"
            })
            
        # Thêm câu lệnh Prompt cuối cùng
        user_content.append({
            "type": "text",
            "text": prompt_text
        })
        
        messages = [{"role": "user", "content": user_content}]
        
        # Gọi API vLLM (Giữ nguyên logic Stream của bạn)
        response_text = ""
        response = client.chat.completions.create(
            model="radiology", 
            messages=messages,
            temperature=0,
            max_tokens=4048,
            stream=False  # Hoặc bỏ dòng này đi vì mặc định là False
        )
        
        # Lấy toàn bộ nội dung text kết quả
        response_text = response.choices[0].message.content

        # Ghi trực tiếp một lần vào file txt
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(response_text)

        return f"[THÀNH CÔNG] Đã xử lý và lưu: {output_file_path}"
        
    except Exception as e:
        # Nếu lỗi, xóa file text hỏng để lần sau chạy lại không bị bỏ qua
        if os.path.exists(output_file_path):
            os.remove(output_file_path)
        return f"[LỖI] Thất bại tại {npy_rel_path}: {str(e)}"

In [5]:
from tqdm import tqdm


# --- TRÌNH ĐIỀU KHIỂN ĐA LUỒNG (MULTITHREADING) ---
def main():
    print("🚀 Đang tải dataset từ Hugging Face...")
    dataset = load_dataset(HF_REPO_ID, split="test")
    # dataset = dataset[:200]
    samples = list(dataset)
    samples = samples[:10]
    total_samples = len(samples)
    print(f"📦 Tổng số ca bệnh cần xử lý: {total_samples}")
    print(f"⚡ Đang chạy song song với tối đa {MAX_CONCURRENT_REQUESTS} requests...")

    # Sử dụng ThreadPoolExecutor để giới hạn tối đa 10 workers
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:
        # Map các task vào luồng xử lý
        future_to_case = {
            executor.submit(process_single_case, sample): sample for sample in samples
        }
        
        # Bọc tqdm quanh as_completed để tự động cập nhật thanh tiến trình
        # desc: Nhãn hiển thị đầu thanh progress
        # total: Tổng số lượng task để tính toán % chính xác
        for future in tqdm(
            as_completed(future_to_case), 
            total=total_samples, 
            desc="Processing CT Scans"
        ):
            try:
                # Vẫn gọi future.result() để bắt các Exception nếu có lỗi xảy ra trong thread
                result_message = future.result()
                
                # Nếu muốn ghi nhận các ca lỗi/bỏ qua, bạn có thể set nó vào postfix của tqdm (tùy chọn)
                # tqdm.write(result_message) # Dùng hàm này nếu vẫn muốn in log mà không làm vỡ thanh tiến trình
                
            except Exception as e:
                # Bắt lỗi không mong muốn từ thread (nếu hàm process_single_case chưa bắt hết)
                tqdm.write(f"❌ Thread lỗi: {e}")

    print("\n🎉 Hoàn thành xử lý toàn bộ dataset!")


if __name__ == "__main__":
    main()

🚀 Đang tải dataset từ Hugging Face...


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.3k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/154 [00:00<?, ? examples/s]

📦 Tổng số ca bệnh cần xử lý: 10
⚡ Đang chạy song song với tối đa 2 requests...


Processing CT Scans: 100%|█████████████████████████████████████████████████████| 10/10 [01:14<00:00,  7.47s/it]


🎉 Hoàn thành xử lý toàn bộ dataset!


In [9]:
import shutil

# Đường dẫn tới folder bạn muốn zip
folder_to_zip = "/root/result"

# Tên file zip sau khi nén (không cần ghi đuôi .zip)
output_zip_name = "result"

# Tiến hành nén thành file 'ten_file_nen.zip'
shutil.make_archive(output_zip_name, "zip", folder_to_zip)

print(f"🎉 Đã zip thành công thành file: {output_zip_name}.zip")

🎉 Đã zip thành công thành file: result.zip
